In [ ]:
import torch
import os
from PIL import Image
from torchvision.transforms.functional import to_tensor, to_pil_image
from tkinter import Tk, filedialog
from basicsr.archs.rrdbnet_arch import RRDBNet
import matplotlib.pyplot as plt

In [ ]:
Tk().withdraw()
img_path = filedialog.askopenfilename(filetypes=[("Image files", "*.jpg *.png *.jpeg")])
if not img_path:
    raise ValueError("No image selected")

img = Image.open(img_path).convert("RGB")
lr_tensor = to_tensor(img).unsqueeze(0)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = RRDBNet(
    num_in_ch=3,
    num_out_ch=3,
    num_feat=64,
    num_block=23,
    num_grow_ch=32,
    scale=4
).to(device)

In [ ]:
weights_path = "weights/RealESRGAN_x4plus.pth"
ckpt = torch.load(weights_path, map_location=device)

if isinstance(ckpt, dict):
    if "params_ema" in ckpt:
        model.load_state_dict(ckpt["params_ema"], strict=True)
    elif "params" in ckpt:
        model.load_state_dict(ckpt["params"], strict=True)
    elif "state_dict" in ckpt:
        model.load_state_dict(ckpt["state_dict"], strict=True)
    else:
        model.load_state_dict(ckpt, strict=True)
else:
    raise RuntimeError("error")

model.eval()

In [ ]:
with torch.no_grad():
    sr = model(lr_tensor.to(device)).squeeze(0).clamp(0, 1).cpu()

sr_img = to_pil_image(sr)
filename = os.path.splitext(os.path.basename(img_path))[0]
out_path = os.path.join(os.path.dirname(img_path), f"{filename}_upscaled.png")
sr_img.save(out_path)

In [ ]:
plt.imshow(sr_img)
plt.axis("off")
plt.title("Upscaled Image")
plt.show()